In [ ]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers==4.56.2 accelerate peft bitsandbytes pandas lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [ ]:
# Core training config.
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    "max_seq_length": 4096,
    "lora_r": 32,
    "lora_alpha": 64,
    "learning_rate": 2e-4,
    "num_train_epochs": 4,
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 50000, #Train on all 50,000 samples in train.csv
    "eval_size": 0.02,
    "output_dir": "/finetunedmodel",
}

SYSTEM_PROMPT = (
    "You are an SVG code generator. Given a description, output only valid SVG code, nothing else. "
    "Only use these elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, "
    "defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, "
    "desc, style, pattern, marker, filter."
    "Keep the final SVG code strictly under 2048 tokens"
)

CONFIG

{'model_name': 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit',
 'max_seq_length': 4096,
 'lora_r': 32,
 'lora_alpha': 64,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 16,
 'gradient_accumulation_steps': 4,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 50000,
 'eval_size': 0.02,
 'output_dir': '/finetunedmodel'}

In [ ]:
#Mound to drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/DL-midterm-2026')

#Read training set, drop id column, convert prompt and svg columns to string type
df = pd.read_csv("train.csv")
df.drop('id', axis = 1)
df = df[['prompt', 'svg']].astype('string')
df.head(1)

Mounted at /content/drive


,prompt,svg
0,The image features two orange squares with a m...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [ ]:
#Begin Exploratory Data Analysis
df['svg'].str.len().describe(percentiles=[.75, .80, .90, .95, .99])

,svg
count,50000.0
mean,2524.277
std,1773.833419
min,91.0
50%,2110.0
75%,3530.0
80%,3947.2
90%,5100.0
95%,6078.0
99%,7514.01


In [ ]:
#The dataset contains floating points with a high degree of decimal precision. This high precision substantially increases token consumption, but
#provides barely any information gain for a 256x256 pixel image. We convert these to floats with 1 decimal place precision.
df['svg'] = df['svg'].str.replace(r'\d+\.\d+', lambda m: f"{float(m.group()):.1f}".rstrip('0').rstrip('.'), regex = True)
df['svg'].str.len().describe()
df['svg'].str.extract(r'(viewBox="[^"]+")')[0].value_counts()

,count
0,
"viewBox=""0 0 200 200""",41383
"viewBox=""0 0 24 24""",2104
"viewBox=""0 0 128 128""",902
"viewBox=""0 0 400 400""",734
"viewBox=""0 0 32 32""",655
...,...
"viewBox=""0 0 26 29.9""",1
"viewBox=""0 -168 512 512""",1
"viewBox=""-22.5 0 301 301""",1


In [ ]:
#Around 80% of the samples have the same viewbox. The rest of them samples will cause spatial confusion during fine tuning. We will drop the other 20% of samples and convert the dominat viewbox to ints
df = df[df['svg'].str.contains(r'viewBox="0 0 200 200"', regex = True, na = False)]
df['svg'].str.extract(r'(viewBox="[^"]+)"')[0].value_counts()
df['svg'].str.len().describe(percentiles=[.75, .80, .90, .95, .99])


,svg
count,41383.0
mean,1141.39811
std,583.598742
min,112.0
50%,1022.0
75%,1489.0
80%,1626.0
90%,2006.0
95%,2302.0
99%,2709.0


In [ ]:
#There is a larger deviation between the average and max value. We need to examine how many svg strings have a length
#that is close to the max. If it is only a few outliers, we can remove them to decrease model's demand on VRAM.
lengths = df['svg'].str.len()

for i in range(1, 10):
    percentile = float(990 + i) / 1000
    lower_bound = lengths.quantile(percentile)
    percentile_lengths = lengths[lengths >= lower_bound]
    print(f"Stats for top {percentile}:")
    print(f"The average length is: {percentile_lengths.mean()}")
    print(f"The min length is: {percentile_lengths.min()}")
    print()

Stats for top 0.991:
The average length is: 2894.2326203208554
The min length is: 2731

Stats for top 0.992:
The average length is: 2910.7455621301774
The min length is: 2750

Stats for top 0.993:
The average length is: 2935.803448275862
The min length is: 2775

Stats for top 0.994:
The average length is: 2959.816
The min length is: 2798

Stats for top 0.995:
The average length is: 2988.341346153846
The min length is: 2842

Stats for top 0.996:
The average length is: 3020.975903614458
The min length is: 2880

Stats for top 0.997:
The average length is: 3062.216
The min length is: 2911

Stats for top 0.998:
The average length is: 3123.0119047619046
The min length is: 2971

Stats for top 0.999:
The average length is: 3233.5476190476193
The min length is: 3058



In [ ]:
#We can keep the size of svg strings to less than 3000 if we cut out the top 0.1 percentile of svg lengths.
#This will help prevent the model from creating excessivley large svg outputs and wasting VRAM resources.
percentile = float(990 + i) / 1000
threshold = lengths.quantile(0.998)
df = df[lengths <= threshold]

#Grab info on the largest svg file for counting tokens later on.
max_row = df.loc[[df['svg'].str.len().idxmax()]]
max_row.head(1)

,prompt,svg
35426,The image contains a green silhouette of a bab...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [ ]:
from unsloth import FastLanguageModel
#The prior operations have cut the average size of the svg strings in half, decreased the max svg size from 15,937 characters to 2971, and standardized the dimensions of the svg viewboxes.
#This will substantially decrease wasteful token consumption for our model. We are now ready to convert these rows into the conversational format that the model expects.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

#Modify dataframe rows for model use
def build_conversations(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["svg"]}
    ]
    formatted_string = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
    return formatted_string

df['text'] = df.apply(build_conversations, axis = 1)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 36eb19fc-66c7-4558-9dae-41dd1008238f)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit/resolve/main/custom_generate/generate.py
Retrying in 1s [Retry 1/5].


In [ ]:
#Convert pandas dataframe to hugging face dataset and split into training and eval sets
hf_dataset = Dataset.from_pandas(df)

split_dataset = hf_dataset.train_test_split(test_size = CONFIG['eval_size'], seed = SEED)
train_ds = split_dataset['train']
eval_ds = split_dataset['test']

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
print(train_ds[:1])

Train rows: 40474
Eval rows: 827
{'prompt': ['A black circular shape with intersecting horizontal and vertical lines forming a grid pattern, set against a solid white background.'], 'svg': ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="#000000" fill-opacity="1"  filling="0" d="M180.6 65.9 A87.8 87.8 0 1 0 18.8 134.1 A87.8 87.8 0 0 0 180.6 65.9 Z M154.8 115.3 A126.2 126.2 0 0 1 133.4 120.7 A223.2 223.2 0 0 0 133.4 79.3 C141.2 80.6 148.5 82.4 154.8 84.7 C163.2 87.8 169.4 91.5 172.5 95.2 A74.2 74.2 0 0 1 172.5 104.8 C169.4 108.5 163.2 112.2 154.8 115.3 Z M104.8 172.5 A74.2 74.2 0 0 1 95.1 172.5 C91.5 169.4 87.7 163.2 84.7 154.8 A126.2 126.2 0 0 1 79.2 133.4 A223.2 223.2 0 0 0 120.7 133.4 A126.2 126.2 0 0 1 115.3 154.8 C112.2 163.2 108.5 169.4 104.8 172.5 Z M100 123.4 C92.6 123.4 85.1 123.1 77.7 122.3 A215.2 215.2 0 0 1 77.7 77.7 A215.2 215.2 0 0 1 122.3 77.7 C123.1 85.1 123.4 92.6 123.4 100 C123.4 107.4 123.1 114.9 122.3 122.3 A210

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

Unsloth 2026.3.17 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
print(f"The maximum SVG string requires {len(tokenizer.encode(max_row['svg'].iloc[0]))} tokens.")

The maximum SVG string requires 2697 tokens.


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import warnings

# Suppress specific Hugging Face FutureWarnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="transformers.*"
)

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

train_result = trainer.train()
train_result

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/40474 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/827 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40,474 | Num Epochs = 1 | Total steps = 633
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Step,Training Loss,Validation Loss
100,0.426300,0.422625
200,0.399400,0.403179
300,0.394000,0.391877
400,0.388000,0.384867
500,0.377600,0.379276
600,0.377400,0.377383


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1691b020-10e8-482d-9fa7-7341bffb3dab)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit/resolve/main/config.json
Retrying in 1s [Retry 1/5].


TrainOutput(global_step=633, training_loss=0.4092728432509195, metrics={'train_runtime': 3514.1897, 'train_samples_per_second': 11.517, 'train_steps_per_second': 0.18, 'total_flos': 7.048848451605627e+17, 'train_loss': 0.4092728432509195, 'epoch': 1.0})

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])

tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

# 1. Zip the folder and save the zip file to Colab's default /content folder
!zip -r /content/finetunedmodel.zip /finetunedmodel

Saved adapter + tokenizer to: /finetunedmodel
updating: finetunedmodel/ (stored 0%)
updating: finetunedmodel/merges.txt (deflated 57%)
updating: finetunedmodel/added_tokens.json (deflated 65%)
updating: finetunedmodel/checkpoint-600/ (stored 0%)
updating: finetunedmodel/checkpoint-600/merges.txt (deflated 57%)
updating: finetunedmodel/checkpoint-600/added_tokens.json (deflated 65%)
updating: finetunedmodel/checkpoint-600/trainer_state.json (deflated 75%)
updating: finetunedmodel/checkpoint-600/optimizer.pt (deflated 13%)
updating: finetunedmodel/checkpoint-600/README.md (deflated 65%)
updating: finetunedmodel/checkpoint-600/tokenizer_config.json (deflated 89%)
updating: finetunedmodel/checkpoint-600/scheduler.pt (deflated 61%)
updating: finetunedmodel/checkpoint-600/training_args.bin (deflated 53%)
updating: finetunedmodel/checkpoint-600/special_tokens_map.json (deflated 67%)
updating: finetunedmodel/checkpoint-600/chat_template.jinja (deflated 71%)
updating: finetunedmodel/checkpoint-